<a href="https://colab.research.google.com/github/Supan-Roy/financial-scam-analysis/blob/main/notebooks/12_integrated_multilevel_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# NOTEBOOK 12: INTEGRATED MULTI-LEVEL INFERENCE
# Cell 1: Environment Setup and Artifact Configuration
# ============================================================

import os
import json
import random
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from google.colab import drive

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Project Directories
# ------------------------------------------------------------
ROOT_DIR = Path("/content/drive/MyDrive/financial-scam-analysis")

DATA_DIR = ROOT_DIR / "data"
MODEL_DIR = ROOT_DIR / "models"
OUTPUT_DIR = ROOT_DIR / "outputs"

BINARY_MODEL_DIR = MODEL_DIR / "distilbert_binary"

MULTILEVEL_MODEL_DIR = (
    MODEL_DIR / "distilbert_multilevel" / "checkpoints"
)

INFERENCE_OUTPUT_DIR = (
    OUTPUT_DIR / "integrated_inference"
)

INFERENCE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Model Checkpoint Paths
# ------------------------------------------------------------
MODEL_PATHS = {
    "binary": BINARY_MODEL_DIR,

    "subtype": (
        MULTILEVEL_MODEL_DIR
        / "subtype_full_combined"
        / "best_model"
    ),

    "intent": (
        MULTILEVEL_MODEL_DIR
        / "intent_full_combined"
        / "best_model"
    ),

    "tactics": (
        MULTILEVEL_MODEL_DIR
        / "tactics_full_combined"
        / "best_model"
    )
}

# ------------------------------------------------------------
# 4. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 5. Device Configuration
# ------------------------------------------------------------
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 6. Inference Configuration
# ------------------------------------------------------------
BASE_MODEL_NAME = "distilbert/distilbert-base-uncased"

MAX_LENGTH = 512
BINARY_THRESHOLD = 0.5
TACTIC_THRESHOLD = 0.5

# ------------------------------------------------------------
# 7. Verify Model Artifacts
# ------------------------------------------------------------
print("=" * 75)
print("INTEGRATED MULTI-LEVEL INFERENCE — ENVIRONMENT")
print("=" * 75)

print(f"\nProject directory: {ROOT_DIR}")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("GPU unavailable — inference will use CPU.")

print("\nMODEL ARTIFACT VERIFICATION")
print("-" * 75)

for model_name, model_path in MODEL_PATHS.items():

    config_path = model_path / "config.json"
    weights_path = model_path / "model.safetensors"

    # Some saved models may use pytorch_model.bin.
    alternative_weights_path = model_path / "pytorch_model.bin"

    exists = (
        model_path.exists()
        and config_path.exists()
        and (
            weights_path.exists()
            or alternative_weights_path.exists()
        )
    )

    print(
        f"{model_name.upper():10s} | "
        f"{'AVAILABLE' if exists else 'MISSING'} | "
        f"{model_path}"
    )

    assert exists, (
        f"Required model artifact missing or incomplete: {model_path}"
    )

print("\n" + "=" * 75)
print("ENVIRONMENT INITIALIZATION COMPLETE")
print("=" * 75)


Mounted at /content/drive
INTEGRATED MULTI-LEVEL INFERENCE — ENVIRONMENT

Project directory: /content/drive/MyDrive/financial-scam-analysis
Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB

MODEL ARTIFACT VERIFICATION
---------------------------------------------------------------------------
BINARY     | AVAILABLE | /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
SUBTYPE    | AVAILABLE | /content/drive/MyDrive/financial-scam-analysis/models/distilbert_multilevel/checkpoints/subtype_full_combined/best_model
INTENT     | AVAILABLE | /content/drive/MyDrive/financial-scam-analysis/models/distilbert_multilevel/checkpoints/intent_full_combined/best_model
TACTICS    | AVAILABLE | /content/drive/MyDrive/financial-scam-analysis/models/distilbert_multilevel/checkpoints/tactics_full_combined/best_model

ENVIRONMENT INITIALIZATION COMPLETE


In [2]:
# ============================================================
# Cell 2: Inference Libraries and Model Configuration
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

from collections import OrderedDict

# ------------------------------------------------------------
# 1. Task Configuration
# ------------------------------------------------------------

TASK_CONFIG = {
    "binary": {
        "num_labels": 2,
        "problem_type": "single_label_classification",
        "max_length": MAX_LENGTH
    },
    "subtype": {
        "num_labels": 13,
        "problem_type": "single_label_classification",
        "max_length": MAX_LENGTH
    },
    "intent": {
        "num_labels": 5,
        "problem_type": "single_label_classification",
        "max_length": MAX_LENGTH
    },
    "tactics": {
        "num_labels": 9,
        "problem_type": "multi_label_classification",
        "max_length": MAX_LENGTH
    }
}

# ------------------------------------------------------------
# 2. Binary Classification Labels
# ------------------------------------------------------------

BINARY_LABELS = {
    0: "legitimate",
    1: "financial_scam"
}

# ------------------------------------------------------------
# 3. Subtype Label Mapping
# ------------------------------------------------------------

SUBTYPE_LABELS = [
    "account_verification",
    "card_transaction_alert",
    "charity_donation",
    "family_emergency_request",
    "investment_crypto",
    "invoice_payment_request",
    "job_task_commission",
    "loan_offer",
    "prize_reward",
    "recovery_support",
    "refund_chargeback",
    "romance_financial_manipulation",
    "tax_payment"
]

# ------------------------------------------------------------
# 4. Intent Label Mapping
# ------------------------------------------------------------

INTENT_LABELS = [
    "gain_account_or_device_access",
    "induce_fraudulent_investment",
    "induce_payment_or_transfer",
    "obtain_credentials_or_codes",
    "obtain_personal_or_financial_information"
]

# ------------------------------------------------------------
# 5. Social-Engineering Tactic Label Mapping
# ------------------------------------------------------------

TACTIC_LABELS = [
    "authority_impersonation",
    "emotional_manipulation",
    "false_legitimacy",
    "fear_or_threat",
    "reward_or_benefit",
    "scarcity",
    "secrecy",
    "trust_exploitation",
    "urgency"
]

# ------------------------------------------------------------
# 6. Configuration Validation
# ------------------------------------------------------------

assert len(SUBTYPE_LABELS) == TASK_CONFIG["subtype"]["num_labels"]
assert len(INTENT_LABELS) == TASK_CONFIG["intent"]["num_labels"]
assert len(TACTIC_LABELS) == TASK_CONFIG["tactics"]["num_labels"]

assert len(set(SUBTYPE_LABELS)) == len(SUBTYPE_LABELS)
assert len(set(INTENT_LABELS)) == len(INTENT_LABELS)
assert len(set(TACTIC_LABELS)) == len(TACTIC_LABELS)

print("=" * 75)
print("INFERENCE CONFIGURATION")
print("=" * 75)

for task, config in TASK_CONFIG.items():
    print(
        f"{task.upper():10s} | "
        f"Labels: {config['num_labels']:2d} | "
        f"Type: {config['problem_type']}"
    )

print("\nSubtype labels:", len(SUBTYPE_LABELS))
print("Intent labels:", len(INTENT_LABELS))
print("Tactic labels:", len(TACTIC_LABELS))

print("\nBinary threshold:", BINARY_THRESHOLD)
print("Tactic threshold:", TACTIC_THRESHOLD)
print("Maximum sequence length:", MAX_LENGTH)

print("\nConfiguration validation successful.")

INFERENCE CONFIGURATION
BINARY     | Labels:  2 | Type: single_label_classification
SUBTYPE    | Labels: 13 | Type: single_label_classification
INTENT     | Labels:  5 | Type: single_label_classification
TACTICS    | Labels:  9 | Type: multi_label_classification

Subtype labels: 13
Intent labels: 5
Tactic labels: 9

Binary threshold: 0.5
Tactic threshold: 0.5
Maximum sequence length: 512

Configuration validation successful.


In [3]:
# ============================================================
# Cell 3: Load Tokenizer and All Trained Models
# ============================================================

# ------------------------------------------------------------
# 1. Load Shared Tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME
)

# ------------------------------------------------------------
# 2. Load Four Trained Models
# ------------------------------------------------------------

models = {}

for task_name, model_path in MODEL_PATHS.items():

    print(f"\nLoading {task_name.upper()} model...")

    model = AutoModelForSequenceClassification.from_pretrained(
        str(model_path)
    )

    model.to(DEVICE)
    model.eval()

    models[task_name] = model

    print(f"  Loaded successfully")
    print(f"  Number of labels: {model.config.num_labels}")
    print(f"  Problem type: {model.config.problem_type}")
    print(f"  Device: {next(model.parameters()).device}")

# ------------------------------------------------------------
# 3. Inspect Saved Label Configurations
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SAVED MODEL LABEL CONFIGURATIONS")
print("=" * 75)

for task_name, model in models.items():

    print(f"\n{task_name.upper()}")
    print("-" * 50)

    print("id2label:")
    print(model.config.id2label)

    print("\nlabel2id:")
    print(model.config.label2id)

# ------------------------------------------------------------
# 4. Validate Model Dimensions
# ------------------------------------------------------------

for task_name, model in models.items():

    expected_labels = TASK_CONFIG[task_name]["num_labels"]

    assert model.config.num_labels == expected_labels, (
        f"{task_name}: expected {expected_labels} labels, "
        f"found {model.config.num_labels}"
    )

assert tokenizer.model_max_length >= MAX_LENGTH

# ------------------------------------------------------------
# 5. Memory Summary
# ------------------------------------------------------------

total_parameters = sum(
    parameter.numel()
    for model in models.values()
    for parameter in model.parameters()
)

print("\n" + "=" * 75)
print("MODEL LOADING COMPLETE")
print("=" * 75)

print(f"Models loaded: {len(models)}")
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size:,}")
print(f"Total parameters across models: {total_parameters:,}")
print(f"Total parameters (millions): {total_parameters / 1e6:.2f}")

if torch.cuda.is_available():
    print(
        f"GPU memory allocated: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Loading BINARY model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  Loaded successfully
  Number of labels: 5
  Problem type: single_label_classification
  Device: cuda:0

Loading TACTICS model...


Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

  Loaded successfully
  Number of labels: 9
  Problem type: multi_label_classification
  Device: cuda:0

SAVED MODEL LABEL CONFIGURATIONS

BINARY
--------------------------------------------------
id2label:
{0: 'LEGITIMATE', 1: 'SCAM'}

label2id:
{'LEGITIMATE': 0, 'SCAM': 1}

SUBTYPE
--------------------------------------------------
id2label:
{0: 'account_verification', 1: 'card_transaction_alert', 2: 'charity_donation', 3: 'family_emergency_request', 4: 'investment_crypto', 5: 'invoice_payment_request', 6: 'job_task_commission', 7: 'loan_offer', 8: 'prize_reward', 9: 'recovery_support', 10: 'refund_chargeback', 11: 'romance_financial_manipulation', 12: 'tax_payment'}

label2id:
{'account_verification': 0, 'card_transaction_alert': 1, 'charity_donation': 2, 'family_emergency_request': 3, 'investment_crypto': 4, 'invoice_payment_request': 5, 'job_task_commission': 6, 'loan_offer': 7, 'prize_reward': 8, 'recovery_support': 9, 'refund_chargeback': 10, 'romance_financial_manipulation': 11

In [4]:
# ============================================================
# Cell 4: Tokenization and Model Input Compatibility Test
# ============================================================

# ------------------------------------------------------------
# 1. Sample Messages
# ------------------------------------------------------------

sample_messages = [
    (
        "Your bank account has been temporarily suspended. "
        "Verify your identity immediately using the secure link."
    ),
    (
        "Thank you for attending today's lecture. "
        "The assignment submission deadline is Friday."
    )
]

# ------------------------------------------------------------
# 2. Tokenization
# ------------------------------------------------------------

encoded_inputs = tokenizer(
    sample_messages,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

encoded_inputs = {
    key: value.to(DEVICE)
    for key, value in encoded_inputs.items()
}

print("=" * 75)
print("TOKENIZATION VALIDATION")
print("=" * 75)

print("Input batch size:", len(sample_messages))
print("Input tensor shape:", encoded_inputs["input_ids"].shape)
print("Attention mask shape:", encoded_inputs["attention_mask"].shape)

# ------------------------------------------------------------
# 3. Run Compatibility Inference
# ------------------------------------------------------------

print("\nMODEL OUTPUT VALIDATION")
print("-" * 75)

with torch.inference_mode():

    for task_name, model in models.items():

        outputs = model(**encoded_inputs)

        logits = outputs.logits

        assert logits.shape == (
            len(sample_messages),
            TASK_CONFIG[task_name]["num_labels"]
        )

        if task_name == "tactics":
            probabilities = torch.sigmoid(logits)
        else:
            probabilities = torch.softmax(logits, dim=-1)

        print(f"\n{task_name.upper()}")
        print("Logits shape:", tuple(logits.shape))
        print("Probability shape:", tuple(probabilities.shape))

        print(
            "Probability range:",
            f"{probabilities.min().item():.4f}",
            "to",
            f"{probabilities.max().item():.4f}"
        )

        if task_name != "tactics":
            probability_sums = probabilities.sum(dim=-1)

            print(
                "Softmax row sums:",
                probability_sums.cpu().numpy().round(4)
            )

            assert torch.allclose(
                probability_sums,
                torch.ones_like(probability_sums),
                atol=1e-5
            )

        else:
            print(
                "Tactic probabilities use independent sigmoid outputs."
            )

# ------------------------------------------------------------
# 4. Token Length Inspection
# ------------------------------------------------------------

print("\nTOKEN LENGTHS")
print("-" * 75)

for message in sample_messages:

    token_count = len(
        tokenizer(
            message,
            truncation=False
        )["input_ids"]
    )

    print(f"{token_count:4d} tokens | {message[:85]}...")

print("\n" + "=" * 75)
print("TOKENIZATION AND MODEL COMPATIBILITY VERIFIED")
print("=" * 75)

TOKENIZATION VALIDATION
Input batch size: 2
Input tensor shape: torch.Size([2, 19])
Attention mask shape: torch.Size([2, 19])

MODEL OUTPUT VALIDATION
---------------------------------------------------------------------------

BINARY
Logits shape: (2, 2)
Probability shape: (2, 2)
Probability range: 0.0000 to 1.0000
Softmax row sums: [1. 1.]

SUBTYPE
Logits shape: (2, 13)
Probability shape: (2, 13)
Probability range: 0.0003 to 0.9908
Softmax row sums: [1. 1.]

INTENT
Logits shape: (2, 5)
Probability shape: (2, 5)
Probability range: 0.0013 to 0.9306
Softmax row sums: [1. 1.]

TACTICS
Logits shape: (2, 9)
Probability shape: (2, 9)
Probability range: 0.0013 to 0.9453
Tactic probabilities use independent sigmoid outputs.

TOKEN LENGTHS
---------------------------------------------------------------------------
  19 tokens | Your bank account has been temporarily suspended. Verify your identity immediately us...
  18 tokens | Thank you for attending today's lecture. The assignment submissio

In [5]:

# ============================================================
# Cell 5: Individual Model Inference Functions
# ============================================================

def tokenize_message(message):
    """Tokenize one message using the shared DistilBERT tokenizer."""

    if not isinstance(message, str) or not message.strip():
        raise ValueError("Input message must be a non-empty string.")

    encoded = tokenizer(
        message,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }


def predict_single_label(task_name, message):
    """
    Run single-label classification.

    Supported tasks:
        binary, subtype, intent
    """

    if task_name not in ["binary", "subtype", "intent"]:
        raise ValueError(f"Unsupported single-label task: {task_name}")

    model = models[task_name]
    inputs = tokenize_message(message)

    with torch.inference_mode():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    predicted_index = int(torch.argmax(probabilities).item())

    label = model.config.id2label[predicted_index]
    confidence = float(probabilities[predicted_index].item())

    all_scores = {
        model.config.id2label[index]: float(probabilities[index].item())
        for index in range(model.config.num_labels)
    }

    return {
        "label": label,
        "confidence": confidence,
        "probabilities": all_scores
    }


def predict_tactics(message, threshold=TACTIC_THRESHOLD):
    """
    Run multilabel social-engineering tactic classification.

    Each tactic receives an independent sigmoid probability.
    Labels are selected using the fixed classification threshold.
    """

    model = models["tactics"]
    inputs = tokenize_message(message)

    with torch.inference_mode():
        outputs = model(**inputs)
        probabilities = torch.sigmoid(outputs.logits)[0]

    tactic_scores = {
        model.config.id2label[index]: float(probabilities[index].item())
        for index in range(model.config.num_labels)
    }

    detected_tactics = [
        {
            "label": label,
            "confidence": score
        }
        for label, score in tactic_scores.items()
        if score >= threshold
    ]

    detected_tactics.sort(
        key=lambda item: item["confidence"],
        reverse=True
    )

    return {
        "detected_tactics": detected_tactics,
        "probabilities": tactic_scores,
        "threshold": threshold
    }


# ------------------------------------------------------------
# Validate function definitions
# ------------------------------------------------------------

assert callable(tokenize_message)
assert callable(predict_single_label)
assert callable(predict_tactics)

print("Inference functions defined successfully.")
print("Single-label tasks: binary, subtype, intent")
print("Multilabel task: tactics")
print("Binary threshold:", BINARY_THRESHOLD)
print("Tactic threshold:", TACTIC_THRESHOLD)


Inference functions defined successfully.
Single-label tasks: binary, subtype, intent
Multilabel task: tactics
Binary threshold: 0.5
Tactic threshold: 0.5


In [6]:
# ============================================================
# Cell 5A: Canonical Preprocessing for Inference
# ============================================================

import re
from typing import Dict, Optional


# ------------------------------------------------------------
# Canonical placeholder mappings
# Reproduced from src/preprocessing/harmonizer.py
# ------------------------------------------------------------

CANONICAL_PLACEHOLDER_MAPPINGS: Dict[str, str] = {
    # URL & Links
    "<URL>": "[URL]",
    "[URL]": "[URL]",
    "<url>": "[URL]",
    "[url]": "[URL]",
    "[LINK REDACTED]": "[URL]",

    # Phone Numbers & Telephony
    "<PHONE>": "[PHONE_NUMBER]",
    "<PHONE_NUMBER>": "[PHONE_NUMBER]",
    "[PHONE_NUMBER]": "[PHONE_NUMBER]",
    "[PHONE]": "[PHONE_NUMBER]",
    "[FAX_NUMBER]": "[PHONE_NUMBER]",

    # Email Addresses
    "<EMAIL>": "[EMAIL_ADDRESS]",
    "<EMAIL_ADDRESS>": "[EMAIL_ADDRESS]",
    "[EMAIL_ADDRESS]": "[EMAIL_ADDRESS]",
    "[EMAIL]": "[EMAIL_ADDRESS]",

    # Personal Names
    "[NAME]": "[PERSON_NAME]",
    "[Name]": "[PERSON_NAME]",
    "<name>": "[PERSON_NAME]",
    "[USERNAME]": "[PERSON_NAME]",
    "[INITIALS]": "[PERSON_NAME]",
    "[Fictional Contact]": "[PERSON_NAME]",

    # Organizations
    "[ORGANIZATION]": "[ORGANIZATION]",
    "[FICTIONAL ORGANIZATION]": "[ORGANIZATION]",
    "[FICTITIOUS SERVICE DESK]": "[ORGANIZATION]",
    "[FICTIONAL SERVICE]": "[ORGANIZATION]",

    # Financial Data
    "<US_BANK_NUMBER>": "[FINANCIAL_DATA]",
    "[FINANCIAL_INFO]": "[FINANCIAL_DATA]",
    "[AMOUNT REDACTED]": "[FINANCIAL_DATA]",
    "[ACCOUNT_NUMBER]": "[FINANCIAL_DATA]",
    "<CREDIT_CARD>": "[FINANCIAL_DATA]",

    # Identifiers
    "<US_DRIVER_LICENSE>": "[IDENTIFIER]",
    "<US_SSN>": "[IDENTIFIER]",
    "<US_PASSPORT>": "[IDENTIFIER]",
    "<UK_NHS>": "[IDENTIFIER]",
    "<MEDICAL_LICENSE>": "[IDENTIFIER]",
    "[REFERENCE_NUMBER]": "[IDENTIFIER]",
    "[CASE REDACTED]": "[IDENTIFIER]",

    # Date & Time
    "<DATE_TIME>": "[DATE_TIME]",
    "[DATE]": "[DATE_TIME]",
    "[date]": "[DATE_TIME]",
    "[TIME]": "[DATE_TIME]",

    # Locations
    "<LOCATION>": "[LOCATION]",
    "[ADDRESS]": "[LOCATION]",

    # Credentials
    "[PASSWORD]": "[CREDENTIAL]",
    "[password]": "[CREDENTIAL]",
    "<password>": "[CREDENTIAL]",

    # Generic Entities
    "<NAMED_ENTITY>": "[NAMED_ENTITY]",
    "<NRP>": "[NRP]",

    # System Artifacts
    "<IP_ADDRESS>": "[SYSTEM_ARTIFACT]",
    "[IP_ADDRESS]": "[SYSTEM_ARTIFACT]",
    "[FILE_NAME]": "[SYSTEM_ARTIFACT]",
    "[FILE_PATH]": "[SYSTEM_ARTIFACT]",
    "[SYMBOL]": "[SYSTEM_ARTIFACT]",
    "[VERSION]": "[SYSTEM_ARTIFACT]",
    "[alternative HTML version deleted]": "[SYSTEM_ARTIFACT]",

    # Retained Domain Tokens
    "[PRODUCT]": "[PRODUCT]",
    "[You]": "[You]"
}


# ------------------------------------------------------------
# Original preprocessing harmonizer
# ------------------------------------------------------------

class PreprocessingHarmonizer:

    def __init__(self, mapping: Optional[Dict[str, str]] = None):
        self.mapping = (
            mapping if mapping is not None
            else CANONICAL_PLACEHOLDER_MAPPINGS
        )

        self._sorted_keys = sorted(
            self.mapping.keys(),
            key=len,
            reverse=True
        )

        self._re_mixed_brackets = re.compile(
            r"<{2,}\s*(?:File:\s*)?\[([A-Za-z0-9_ -]+?)\]\s*>{2,}"
        )

        self._re_multi_square_open = re.compile(
            r"\[{2,}([A-Za-z0-9_ -]+?)\]+"
        )

        self._re_multi_square_close = re.compile(
            r"\[+([A-Za-z0-9_ -]+?)\]{2,}"
        )

        self._re_multi_angle_open = re.compile(
            r"<{2,}([A-Za-z0-9_ -]+?)>+"
        )

        self._re_multi_angle_close = re.compile(
            r"<+([A-Za-z0-9_ -]+?)>{2,}"
        )

    def clean_nested_brackets(self, text: str) -> str:

        if not isinstance(text, str) or not text:
            return "" if text is None else str(text)

        text = self._re_mixed_brackets.sub(r"[\1]", text)
        text = self._re_multi_square_open.sub(r"[\1]", text)
        text = self._re_multi_square_close.sub(r"[\1]", text)
        text = self._re_multi_angle_open.sub(r"<\1>", text)
        text = self._re_multi_angle_close.sub(r"<\1>", text)

        return text

    def normalize_text(self, text: str) -> str:

        if not isinstance(text, str) or not text:
            return "" if text is None else str(text)

        text = self.clean_nested_brackets(text)

        for token in self._sorted_keys:
            target = self.mapping[token]

            if token in text:
                text = text.replace(token, target)

        return text


# ------------------------------------------------------------
# Initialize harmonizer
# ------------------------------------------------------------

harmonizer = PreprocessingHarmonizer()


# ------------------------------------------------------------
# Override tokenizer with training-consistent preprocessing
# ------------------------------------------------------------

def tokenize_message(message):

    if not isinstance(message, str) or not message.strip():
        raise ValueError("Input message must be a non-empty string.")

    normalized_message = harmonizer.normalize_text(message)

    encoded = tokenizer(
        normalized_message,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

assert callable(harmonizer.normalize_text)
assert callable(tokenize_message)

test_message = "URGENT! Verify your <EMAIL> and <<[FILE_NAME]>> immediately."

normalized_test = harmonizer.normalize_text(test_message)

print("Preprocessing harmonizer initialized successfully.")
print("\nOriginal message:")
print(test_message)

print("\nNormalized message:")
print(normalized_test)

print("\nTokenizer output shape:")
print(tokenize_message(test_message)["input_ids"].shape)

Preprocessing harmonizer initialized successfully.

Original message:
URGENT! Verify your <EMAIL> and <<[FILE_NAME]>> immediately.

Normalized message:
URGENT! Verify your [EMAIL_ADDRESS] and [SYSTEM_ARTIFACT] immediately.

Tokenizer output shape:
torch.Size([1, 19])


In [7]:
# ============================================================
# Cell 5B: Verify Normalization Against Existing Dataset
# ============================================================

import pandas as pd

TRAIN_PATH = DATA_DIR / "splits" / "train.csv"

sample_df = pd.read_csv(
    TRAIN_PATH,
    usecols=["original_text", "normalized_text"],
    nrows=1000
)

mismatches = []

for index, row in sample_df.iterrows():
    generated = harmonizer.normalize_text(row["original_text"])

    if generated != row["normalized_text"]:
        mismatches.append({
            "index": index,
            "original_text": row["original_text"],
            "expected": row["normalized_text"],
            "generated": generated
        })

print("Records checked:", len(sample_df))
print("Normalization mismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatch:")
    print(mismatches[0])
else:
    print("\nPASS: All sampled records match the existing normalized_text.")

Records checked: 1000
Normalization mismatches: 0

PASS: All sampled records match the existing normalized_text.


In [9]:

# ============================================================
# Cell 6: End-to-End Conditional Inference Engine
# ============================================================

def analyze_message(message):
    """
    Analyze one digital communication through the complete
    multi-level financial scam detection pipeline.

    Level 1:
        Binary legitimate/scam classification.

    Level 2 (conditional):
        Subtype classification.
        Fraudulent intent classification.
        Social-engineering tactics recognition.

    Second-level models execute only when Level 1 predicts scam.
    """

    if not isinstance(message, str) or not message.strip():
        raise ValueError("Input message must be a non-empty string.")

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # Level 1: Binary Detection
    # --------------------------------------------------------

    binary_result = predict_single_label(
        task_name="binary",
        message=message
    )

    scam_probability = binary_result["probabilities"]["SCAM"]

    is_scam = scam_probability >= BINARY_THRESHOLD

    result = {
        "input_message": message,
        "classification": {
            "label": "financial_scam" if is_scam else "legitimate",
            "confidence": (
                scam_probability
                if is_scam
                else 1.0 - scam_probability
            ),
            "scam_probability": scam_probability,
            "legitimate_probability": (
                binary_result["probabilities"]["LEGITIMATE"]
            ),
            "threshold": BINARY_THRESHOLD
        },
        "second_level_analysis": None,
        "routing": {
            "level_1_completed": True,
            "level_2_executed": False
        }
    }

    # --------------------------------------------------------
    # Stop if the message is classified as legitimate
    # --------------------------------------------------------

    if not is_scam:

        result["routing"]["decision"] = (
            "Terminated after legitimate classification"
        )

    # --------------------------------------------------------
    # Level 2: Multi-Level Scam Analysis
    # --------------------------------------------------------

    else:

        subtype_result = predict_single_label(
            task_name="subtype",
            message=message
        )

        intent_result = predict_single_label(
            task_name="intent",
            message=message
        )

        tactics_result = predict_tactics(message)

        result["second_level_analysis"] = {
            "subtype": subtype_result,
            "fraudulent_intent": intent_result,
            "social_engineering_tactics": tactics_result
        }

        result["routing"]["level_2_executed"] = True

        result["routing"]["decision"] = (
            "Routed to all three second-level classifiers"
        )

    # --------------------------------------------------------
    # Execution Time
    # --------------------------------------------------------

    result["execution_time_ms"] = round(
        (time.perf_counter() - start_time) * 1000,
        3
    )

    return result


print("End-to-end inference engine created successfully.")

print("\nPipeline routing:")
print("1. Incoming message")
print("2. Binary classification")
print("3. Legitimate -> Stop")
print("4. Scam -> Subtype + Intent + Tactics")
print("5. Unified structured result")


End-to-end inference engine created successfully.

Pipeline routing:
1. Incoming message
2. Binary classification
3. Legitimate -> Stop
4. Scam -> Subtype + Intent + Tactics
5. Unified structured result


In [12]:

# ============================================================
# Cell 7: Pipeline Routing Smoke Test
# ============================================================

test_messages = {
    "legitimate_example": (
    "Subject: Notice Regarding Semester Registration\n"
    "Dear Students,\n\n"
    "The Department of Computer Science and Engineering "
    "would like to inform you that registration for the "
    "upcoming semester will begin on December 1, 2026.\n\n"
    "Students are requested to review their course selections "
    "and consult their academic advisors if they have any "
    "questions regarding prerequisites or credit requirements.\n\n"
    "The detailed registration schedule and relevant "
    "instructions are available on the university's official website.\n\n"
    "Please contact the department office during regular "
    "working hours for further assistance.\n\n"
    "Best regards,\n"
    "Department of Computer Science and Engineering"
),

    "scam_example": (
        "URGENT: Your bank account will be suspended today. "
        "Verify your account immediately using the link below "
        "to avoid permanent account restriction. "
        "Do not share this message with anyone."
    )
}

smoke_test_results = {}

for example_name, message in test_messages.items():

    print("\n" + "=" * 85)
    print(f"TEST: {example_name.upper()}")
    print("=" * 85)

    result = analyze_message(message)
    smoke_test_results[example_name] = result

    print("\nMessage:")
    print(message)

    print("\nLevel 1 Result:")
    print("Classification:", result["classification"]["label"])
    print(
        "Confidence:",
        f"{result['classification']['confidence']:.4f}"
    )
    print(
        "Scam probability:",
        f"{result['classification']['scam_probability']:.4f}"
    )

    print("\nRouting:")
    print("Level 2 executed:", result["routing"]["level_2_executed"])
    print("Decision:", result["routing"]["decision"])

    if result["second_level_analysis"] is not None:

        analysis = result["second_level_analysis"]

        print("\nLevel 2 Results:")

        print(
            "Subtype:",
            analysis["subtype"]["label"],
            f"({analysis['subtype']['confidence']:.4f})"
        )

        print(
            "Fraudulent intent:",
            analysis["fraudulent_intent"]["label"],
            f"({analysis['fraudulent_intent']['confidence']:.4f})"
        )

        print("Detected tactics:")

        tactics = analysis["social_engineering_tactics"]["detected_tactics"]

        if tactics:
            for tactic in tactics:
                print(
                    f"  - {tactic['label']}: "
                    f"{tactic['confidence']:.4f}"
                )
        else:
            print("  No tactics exceeded the classification threshold.")

    print(
        "\nExecution time:",
        f"{result['execution_time_ms']:.2f} ms"
    )


# ------------------------------------------------------------
# Validate conditional routing consistency
# ------------------------------------------------------------

for example_name, result in smoke_test_results.items():

    predicted_scam = (
        result["classification"]["label"] == "financial_scam"
    )

    assert result["routing"]["level_1_completed"] is True

    assert (
        result["routing"]["level_2_executed"] == predicted_scam
    )

    assert (
        (result["second_level_analysis"] is not None)
        == predicted_scam
    )

    if predicted_scam:
        assert "subtype" in result["second_level_analysis"]
        assert "fraudulent_intent" in result["second_level_analysis"]
        assert "social_engineering_tactics" in result["second_level_analysis"]

print("\n" + "=" * 85)
print("ALL ROUTING CONSISTENCY CHECKS PASSED")
print("=" * 85)



TEST: LEGITIMATE_EXAMPLE

Message:
Subject: Notice Regarding Semester Registration
Dear Students,

The Department of Computer Science and Engineering would like to inform you that registration for the upcoming semester will begin on December 1, 2026.

Students are requested to review their course selections and consult their academic advisors if they have any questions regarding prerequisites or credit requirements.

The detailed registration schedule and relevant instructions are available on the university's official website.

Please contact the department office during regular working hours for further assistance.

Best regards,
Department of Computer Science and Engineering

Level 1 Result:
Classification: legitimate
Confidence: 0.9945
Scam probability: 0.0055

Routing:
Level 2 executed: False
Decision: Terminated after legitimate classification

Execution time: 24.30 ms

TEST: SCAM_EXAMPLE

Message:
URGENT: Your bank account will be suspended today. Verify your account immediatel